# SURF 2026 Week 1 Validation

This notebook validates the benchmark harness, imports the baseline modules, checks the real smart-grid telemetry dataset, and runs quick AES/RSA baselines using telemetry-derived payload bytes.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
import pandas as pd

from benchmarks.runner import BenchmarkRunner
from src.classical.aes_baseline import benchmark_aes_baselines
from src.classical.rsa_baseline import benchmark_rsa_baselines
from src.smartgrid.workloads import (
    build_payloads,
    load_smart_grid_dataset,
    summarize_smart_grid_dataset,
)

## Dataset sanity check

In [ ]:
dataset = load_smart_grid_dataset(PROJECT_ROOT / "data" / "smart_grid_dataset.csv")
dataset_summary = summarize_smart_grid_dataset(
    dataset,
    path=PROJECT_ROOT / "data" / "smart_grid_dataset.csv",
)

pd.DataFrame([dataset_summary.to_row()]).T.rename(columns={0: "value"})

In [ ]:
dataset.head()

## Telemetry-derived benchmark payloads

In [ ]:
payload_sizes = (64, 128, 512)
payloads = build_payloads(dataset, payload_sizes)
pd.DataFrame(
    {
        "payload_bytes": list(payloads.keys()),
        "actual_bytes": [len(payload) for payload in payloads.values()],
        "preview": [payload[:48].decode("utf-8", errors="replace") for payload in payloads.values()],
    }
)

## Week 1 AES/RSA baseline checks

In [ ]:
runner = BenchmarkRunner(results_dir=PROJECT_ROOT / "benchmarks" / "results" / "workstation")
payload_factory = lambda size: payloads[size]

rsa_records = benchmark_rsa_baselines(
    runner,
    trials=10,
    payload_sizes=(64, 128),
    key_sizes=(2048,),
    payload_factory=payload_factory,
    export_path="week1_rsa_validation.csv",
)

aes_records = benchmark_aes_baselines(
    runner,
    trials=10,
    payload_sizes=(64, 512),
    key_sizes=(128, 256),
    payload_factory=payload_factory,
    export_path="week1_aes_validation.csv",
)

In [ ]:
records = [record.to_row() for record in rsa_records + aes_records]
summary = pd.DataFrame(records)
summary[["scheme", "mode", "key_size", "payload_bytes", "operation", "mean_ms", "median_ms", "p95_ms"]]

In [ ]:
comparison = (
    summary.groupby(["scheme", "operation"])[["mean_ms", "median_ms", "p95_ms"]]
    .mean()
    .sort_values(["scheme", "operation"])
)
comparison